<a id='Intro'></a>
# Introduction. What is MLOps?
MLOps stands for Machine Learning Operations. MLOps is a core function of Machine Learning engineering, focused on streamlining the process of taking machine learning models to production, and then maintaining and monitoring them. MLOps is a collaborative function, often comprising data scientists, devops engineers, and IT. <br>

&#x20;*MLOps covers the full machine learning lifecycle, from data sourcing and model training to deployment and monitoring (illustration of an MLOps cycle).*
![](https://cms.databricks.com/sites/default/files/inline-images/mlops-components.png) <br>


One foundational component of MLOps is **experiment tracking**. When developing an ML model, especially NLP models which require numerous experiments with different hyperparameters, training data subsets, or model architectures, it becomes difficult to remember what was tried and which experiment produced the best results. Experiment tracking is the process of logging all the relevant metadata from each run – datasets used, model parameters, code versions, metrics, etc. – so that you can reproduce and compare experiments easily. Proper experiment tracking lets you answer questions like “Which learning rate gave the highest F1 score on the validation set?” or “Which version of the data was this model trained on?” without guesswork. It not only aids reproducibility but also helps in early stopping of poor runs by monitoring metrics in real time.

**Tools for Experiment Tracking:** A popular tool for experiment tracking is **Weights & Biases (wandb)**, which provides a convenient Python API to log metrics, model artifacts, and even custom plots during training. Other tools in this space include **MLflow**, **Neptune.ai**, and **TensorBoard**, but Weights & Biases has gained wide adoption for its powerful dashboard and easy integration. For example, using `wandb` you can track hyperparameters and metrics across runs and visualize them in a web UI.

Beyond tracking experiments, general MLOps for NLP also involves: model version control (managing different trained model checkpoints and deploying the right one), data versioning (since NLP models are very data-dependent, keeping track of which data and preprocessing was used), continuous integration and delivery (CI/CD) for ML (automating the retraining and deployment pipeline), and monitoring deployed models for **data drift** or **performance drift**. For instance, an NLP model might slowly become less accurate if user language shifts over time; an MLOps pipeline would catch this via monitoring and trigger a retraining or alert. In summary, MLOps provides the framework to take an NLP model from a data scientist’s notebook to a robust, scalable production service, while maintaining quality and agility.

# Quantization Techniques and Formats

![](https://developer-blogs.nvidia.com/wp-content/uploads/2021/07/qat-training-precision.png)
![](https://developer-blogs.nvidia.com/wp-content/uploads/2021/07/8-bit-signed-integer-quantization.png)

## GPTQ. Why Post-Training Quantization?

Modern LLMs often exceed tens of GB in FP16, which limits edge deployment and even server-side throughput.  *Weight-only post-training quantization (PTQ)* compresses a **frozen** model **after** training—no gradient updates, no data-intensive fine-tuning—by mapping full-precision weights \$w\$ to lower-bit representations \$\hat w\$ while trying to keep perplexity intact. GPTQ (`Generalized Post-Training Quantization`) is one of the most accurate PTQ algorithms for LLMs; it routinely reaches 4-bit (and even 3-bit) weight precision with \$<1\$ ppl loss on WikiText2 for models up to 175 B params. ([arXiv][1], [arXiv][2])

### Core Ideas

* **Block-wise Optimal Brain Quantizer (OBQ).** GPTQ borrows the *Optimal Brain Surgeon* pruning formalism. GPTQ treats each linear/conv block independently and minimises the *local* (layer-wise) reconstruction loss  
$$
\mathcal{L}(W_q)=\tfrac12\bigl\|(W-W_q)X\bigr\|_{F}^{2},
$$  
* **Group-wise Weight Sharing.** Weights are partitioned into groups of size \$g\$ (typical \$g=128\$). All weights in a group share the same scale \$S\$ and zero-point \$z\$, drastically reducing metadata while keeping accuracy high. ([GitHub][4], [Reddit][5])
* **Fast Greedy Solvers.** GPTQ solves the optimization in a single pass per group (\$O(g^2)\$) and supports parallel GPU kernels; quantizing Llama-65B to 4-bit now takes ≈ 20 min on 4×A100. ([Lightning AI][7])


### Example – Quantizing Llama 3 to 4-bit with `GPTQModel`

```python
from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"
quant_path = "Llama-3.2-1B-Instruct-gptqmodel-4bit"

calibration_dataset = load_dataset(
    "allenai/c4",
    data_files="en/c4-train.00001-of-01024.json.gz",
    split="train"
  ).select(range(1024))["text"]

quant_config = QuantizeConfig(bits=4, group_size=128)

model = GPTQModel.load(model_id, quant_config)

# increase `batch_size` to match gpu/vram specs to speed up quantization
model.quantize(calibration_dataset, batch_size=1)

model.save(quant_path)
```

### References

1. Frantar et al., “GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers”, arXiv 2210.17323. ([arXiv][1])
2. *auto-gptq* GitHub repository (docs & scripts). ([GitHub][8])
3. Medium TDS blog, “4-bit Quantization with GPTQ” (conceptual tutorial). ([Medium][3])
4. Hugging Face model card: `gemma-2b-GPTQ`. ([Hugging Face][11])
5. Lightning AI Studio notebook on GPTQ PTQ. ([Lightning AI][7])
6. AutoGPTQ `perplexity.py` benchmarking example. ([GitHub][10])
7. Reddit explainer on group size & act-order. ([GitHub][4])
8. Hugging Face doc: `quantization` API options (`group_size`, `damp_percent`, …). ([Hugging Face][9])
9. TheBloke model README clarifying act-order effects. ([Hugging Face][6])
10. Discussion of group size vs VRAM trade-off. ([Reddit][5])

[1]: https://arxiv.org/abs/2210.17323?utm_source=chatgpt.com "GPTQ: Accurate Post-Training Quantization for Generative Pre ..."
[2]: https://arxiv.org/pdf/2210.17323?utm_source=chatgpt.com "[PDF] gptq: accurate post-training quantization - arXiv"
[3]: https://medium.com/towards-data-science/4-bit-quantization-with-gptq-36b0f4f02c34?utm_source=chatgpt.com "4-bit Quantization with GPTQ | TDS Archive - Medium"
[4]: https://github.com/pytorch/executorch/issues/3559?utm_source=chatgpt.com "what's the meaning of \"Groupwise 4-bit (128)\" · Issue #3559 - GitHub"
[5]: https://www.reddit.com/r/LocalLLaMA/comments/12rtg82/what_is_group_size_128_and_why_do_30b_models_give/?utm_source=chatgpt.com "What is group size 128 and why do 30b models give the option to ..."
[6]: https://huggingface.co/TheBloke/CodeLlama-13B-GPTQ/blob/gptq-4bit-128g-actorder_True/README.md?utm_source=chatgpt.com "TheBloke/CodeLlama-13B-GPTQ at gptq-4bit-128g-actorder_True"
[7]: https://lightning.ai/cosmo3769/studios/post-training-quantization-to-gptq-format-and-evaluation?utm_source=chatgpt.com "Post-Training Quantization to GPTQ format and Evaluation"
[8]: https://github.com/AutoGPTQ/AutoGPTQ?utm_source=chatgpt.com "AutoGPTQ/AutoGPTQ: An easy-to-use LLMs quantization ... - GitHub"
[9]: https://huggingface.co/docs/transformers/en/main_classes/quantization?utm_source=chatgpt.com "Quantization - Hugging Face"
[10]: https://github.com/PanQiWei/AutoGPTQ/blob/main/examples/benchmark/perplexity.py?utm_source=chatgpt.com "AutoGPTQ/examples/benchmark/perplexity.py at main - GitHub"
[11]: https://huggingface.co/TechxGenus/gemma-2b-GPTQ?utm_source=chatgpt.com "TechxGenus/gemma-2b-GPTQ - Hugging Face"
[12]: https://huggingface.co/TechxGenus/CodeGemma-7b-GPTQ?utm_source=chatgpt.com "TechxGenus/CodeGemma-7b-GPTQ - Hugging Face"
[13]: https://huggingface.co/TheBloke/Llama-2-13B-GPTQ/blob/gptq-8bit-128g-actorder_True/README.md?utm_source=chatgpt.com "TheBloke/Llama-2-13B-GPTQ at gptq-8bit-128g-actorder_True"


## Activation-aware Weight Quantization (AWQ)

AWQ is a **post-training, weight-only, 4-bit quantization method** that explicitly leverages *activation statistics* to decide **which output channels are worth protecting** before rounding weights. Proposed by Lin *et al.* in 2023, it delivers ≈ 0.1 – 0.3 pp perplexity drop on Llama-2/3, Code Llama and VILA while giving **3–4 × decoding speed-ups** on edge GPUs when paired with the TinyChat runtime.([arXiv][1], [MIT HAN Lab][2])

### Why do we need a new PTQ method?

* **GPTQ** minimises quantisation error using a second-order Hessian approximation but can *over-fit* its tiny calibration set, hurting generalisation to new domains.([arXiv][3], [bitbasti.com][4])
* **AWQ** instead asks: *“Which weight channels contribute most to the output variance given real activations?”* Only \~1 % of channels matter; scaling them up prior to quantisation is enough to keep accuracy.([arXiv][1], [Medium][5])

## Mathematical intuition

Let a linear layer output

$$
\mathbf{y} = W\mathbf{x},
$$

where \$\mathbf{x}\sim\mathcal{D}\$ is the activation distribution seen during calibration.

AWQ minimises

$$
\min_{\tilde W,\; \boldsymbol\alpha}\;
\mathbb{E}_{\mathbf{x}\sim\mathcal{D}}
\!\left[\lVert (\,\mathrm{diag}(\boldsymbol\alpha)\tilde W - W)\,\mathbf{x}\rVert_2^{\,2}\right],
$$

where \*\*\$\boldsymbol\alpha\$ is a *vector of learn-able, per-output-channel scaling factors* (typically \$\alpha\_c\ge1\$ for only the ≈1 % most “salient’’ channels), and \*\*\$\operatorname{diag}(\boldsymbol\alpha)\$ is the diagonal matrix that places those scalars on its main diagonal so that left-multiplication \$\operatorname{diag}(\boldsymbol\alpha),\tilde W\$ multiplies—i.e. *rescales*—each row of the quantised weight matrix \$\tilde W\$ by the corresponding \$\alpha\_c\$ before the matrix is used in the linear layer.

subject to each row of \$\tilde W\$ being **k-bit** quantised and \$\alpha\_c\ge1\$ a learned scale for “salient” output channel \$c\$. Channels with large

$$
\text{Var}_\mathbf{x}\bigl((W\mathbf{x})_c\bigr)
$$

are deemed salient; only those get a scale factor, avoiding mixed precision. The optimisation is solved greedily without back-prop, so calibration is **fast and data-independent**.([arXiv][1], [GitHub][6])

## Practical pipeline

1. **Collect activations** on 128–256 random prompts (no labels required).
2. **Rank channels** by output variance.
3. **Assign scales** \$\alpha\_c\$ to the top 1 %.
4. **Uniform-quantise** the scaled weights to 4-bit.
5. **Pack** weights in an AWQ-aware kernel (e.g. TinyChat or vLLM-AWQ).([MIT HAND Lab][7], [GitHub][8])

### One-liner with *AutoAWQ*

```bash
pip install autoawq
# quantise Meta Llama-2-7B in 4-bit
python -m awq \
       --model meta-llama/Llama-2-7b-hf \
       --out ./llama2-7b-awq \
       --wbits 4 --groupsize 128 \
       --nsamples 256 --seqlen 128
```

The resulting directory can be served by TinyChat, LMDeploy or `autoawq`’s own loader and fits in ≈ 3 GB VRAM.([GitHub][9])

> **Pre-quantised checkpoints**
>
> * Llama-2-7B-base-4bit-AWQ (HuggingFace)([Hugging Face][10])


### Further Reading

* Lin *et al.* “AWQ: Activation-aware Weight Quantization” (MLSys ’24 best paper).([arXiv][1])
* AutoAWQ GitHub & model zoo for dozens of AWQ checkpoints.([GitHub][9])
* Frantar *et al.* “GPTQ: Accurate Post-Training Quantization for GPT-style Models”.([arXiv][3])
* Tutorial “LLM Quantization with GPTQ, AWQ and BitsAndBytes” (Towards AI).([Towards AI][17])
* Blog “Deep dive into GPTQ and AWQ” (Medium).([Medium][18])


[1]: https://arxiv.org/abs/2306.00978?utm_source=chatgpt.com "AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration"
[2]: https://hanlab.mit.edu/?754e4700_page=2&7b5b9555_page=2&utm_source=chatgpt.com "Efficient AI Computing, Transforming the Future. - MIT HAN Lab"
[3]: https://arxiv.org/pdf/2210.17323?utm_source=chatgpt.com "[PDF] gptq: accurate post-training quantization - arXiv"
[4]: https://bitbasti.com/blog/why-you-should-not-trust-benchmarks?utm_source=chatgpt.com "Why LLM Benchmarks Can Be Misleading - AWQ vs. GPTQ - bitbasti"
[5]: https://medium.com/friendliai/understanding-activation-aware-weight-quantization-awq-boosting-inference-serving-efficiency-in-10bb0faf63a8?utm_source=chatgpt.com "Understanding Activation-Aware Weight Quantization (AWQ ..."
[6]: https://github.com/mit-han-lab/llm-awq?utm_source=chatgpt.com "AWQ: Activation-aware Weight Quantization for LLM ... - GitHub"
[7]: https://mit-han-lab.github.io/TinyChatEngine/?utm_source=chatgpt.com "TinyChatEngine: On-Device LLM/VLM Inference Library"
[8]: https://github.com/vllm-project/vllm/issues/1853?utm_source=chatgpt.com "Speed between gptq w4a16 and awq w4a16? · Issue #1853 · vllm ..."
[9]: https://github.com/casper-hansen/AutoAWQ?utm_source=chatgpt.com "casper-hansen/AutoAWQ - GitHub"
[10]: https://huggingface.co/TitanML/llama2-7b-base-4bit-AWQ?utm_source=chatgpt.com "TitanML/llama2-7b-base-4bit-AWQ - Hugging Face"
[11]: https://huggingface.co/TechxGenus/gemma-2b-AWQ?utm_source=chatgpt.com "TechxGenus/gemma-2b-AWQ - Hugging Face"
[12]: https://www.e2enetworks.com/blog/which-quantization-method-is-best-for-you-gguf-gptq-or-awq?utm_source=chatgpt.com "Which Quantization Method Is Best for You?: GGUF, GPTQ, or AWQ"
[13]: https://bitbasti.com/blog/faster-llms-with-quantization?utm_source=chatgpt.com "How to get faster inference times with quantization - bitbasti"
[14]: https://friendli.ai/blog/quantization-reduce-llm-size?utm_source=chatgpt.com "Which Quantization to Use to Reduce the Size of LLMs? - FriendliAI"
[15]: https://huggingface.co/TechxGenus/gemma-2b-GPTQ?utm_source=chatgpt.com "TechxGenus/gemma-2b-GPTQ - Hugging Face"
[16]: https://github.com/AutoGPTQ/AutoGPTQ?utm_source=chatgpt.com "AutoGPTQ/AutoGPTQ: An easy-to-use LLMs quantization ... - GitHub"
[17]: https://towardsai.net/p/artificial-intelligence/llm-quantization-quantize-model-with-gptq-awq-and-bitsandbytes?utm_source=chatgpt.com "LLM Quantization: Quantize Model with GPTQ, AWQ, and Bitsandbytes"
[18]: https://medium.com/%40kimdoil1211/speeding-up-large-language-models-a-deep-dive-into-gptq-and-awq-quantization-0bb001eaabd4?utm_source=chatgpt.com "A Deep Dive into GPTQ and AWQ Quantization | by Doil Kim | May ..."


# CPU Offloading Strategies

Even with quantization, some LLMs are too large to fit entirely on a single GPU. **Offloading** is a strategy to utilize CPU (and even disk) memory in conjunction with GPU memory to handle models that exceed the GPU’s capacity. The idea is to **split the model’s weights** across available devices and move parts into GPU memory only when needed for a forward pass.

One simple form of offloading is provided by Hugging Face Accelerate. By loading a model with `device_map="auto"`, the library will automatically split the model across GPU(s) and CPU. It prioritizes GPU memory for as much of the model as possible, then uses CPU RAM for the rest, and if even RAM is insufficient it can use disk (this hierarchy ensures the model *works*, albeit with increasing latency as you offload to slower memory). Offloading means at inference time, layers that reside on CPU will have their computations done on CPU or moved temporarily to GPU, incurring a speed penalty but enabling the model to run. For instance, if you attempt to load a 30B model on a 16 GB GPU, Accelerate might load 50% of layers on the GPU and 50% on the CPU. It ensures that as each layer is needed for the forward pass, its weights are on the GPU (moving them just in time) and then possibly freeing memory after usage. This way, the GPU never holds the entire model at once.

**Example code for offloading:** (This is a conceptual example; we won’t execute it due to time and resource constraints)

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model_name = "facebook/opt-6.7b"  # 6.7B OPT model as an example
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto", offload_folder="offload")
```

In this snippet, setting `device_map="auto"` would distribute the OPT 6.7B model’s layers across the GPU and CPU automatically. The `offload_folder` is a path where, if needed, it can offload some weights to disk. HuggingFace’s documentation notes that with these settings, *“the GPU(s) are filled first, then CPU, then disk”* to make big model inference possible.


On the CPU side, recall that **llama.cpp** natively supports CPU+GPU hybrid inference. You can configure how many layers to run on the GPU vs CPU. This is a form of manual offloading – e.g., “run the first N layers on GPU, the rest on CPU” – which can accelerate inference if your GPU can at least take a portion of the model. In summary, offloading is about using every memory tier (GPU VRAM, CPU RAM, disk) to accommodate large models. The cost is increased latency, especially if disk swapping is involved. As an LLM practitioner, you would use offloading when you need a model that’s slightly too large for your GPU, and you’re willing to trade some speed to get it running. It’s a valuable technique to be aware of for maximizing the utilization of available hardware.

# Model Distillation (Compressing LLMs)

While quantization and offloading deal with deployment optimizations, **model distillation** is a training-time technique to make a smaller model that reproduces the behavior of a larger model. The idea, originating from Hinton et al.’s work on Knowledge Distillation (2015), is to use a large “teacher” model to generate training signals (soft targets) for a smaller “student” model. The student is trained to match the teacher’s output distributions, which often allows it to achieve much better performance than if it were trained from scratch on the original data.

In the context of LLMs, distillation is extremely attractive: if you have a 70B parameter model that performs wonderfully but is too heavy to serve, you’d want to compress its knowledge into, say, a 7B or 13B model that is cheaper and faster. Distillation can be done on general language modeling (having the student predict the next token probabilities that the teacher would) or on specific tasks (having the student imitate the teacher’s answers on a QA task, for example). It’s a complex process (you need to generate a lot of teacher outputs as training data, and carefully balance loss terms), but has been used in practice to create some notable models.

For example, **Google’s Gemma 2** (2024) project introduced a family of small models (2B, 9B, etc.) which were trained with knowledge distillation from a larger model instead of the standard next-token prediction loss. The result was that these distilled 2B and 9B models achieved state-of-the-art performance *for their size*, even rivaling other models that were 2–3× larger in parameter count. This demonstrates the power of distillation: a well-distilled model can punch above its weight. Another classic example is **DistilBERT** in the pre-LLM era – a distilled version of BERT-base which is 40% smaller and 60% faster, yet retains about 97% of BERT’s performance on language understanding tasks. This made BERT practical to use in real-time applications.

How does one perform distillation on an LLM? At a high level:

1. You take the large teacher model (e.g., a LLaMA3-70B if it existed) and a smaller student model (e.g., LLaMA3-7B or 13B).
2. You generate a large set of inputs – these could be a mixture of text from the training distribution, plus special prompts if focusing on an instruction-following behavior.
3. You run the teacher model to get outputs for these inputs. The outputs could be the probability distribution over the next token at each position (for language modeling distillation) or the final generated text (for conversation distillation).
4. You train the student model on this data, using a loss that makes the student’s outputs close to the teacher’s outputs. If using token distributions, one common loss is the KL-divergence between the teacher’s softmax output and the student’s output at each position (this is the “soft target” approach Hinton described). You can also combine it with the regular loss on the true label (if supervised data is available) – known as a “teacher-student” training regime.

![](https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/Knowledge-Distillation_4.png?resize=900%2C356&ssl=1)
- Forward Pass through the Teacher network : Pass the data through the teacher network to get all intermediate outputs and then apply data augmentation (if any) to the same.
- Backpropagation through the Student Network : Now use the outputs from the teacher network and the correspondence relation to backpropagate error in the student network, so that the student network can learn to replicate the behavior of the teacher network. 


> **TODO**: nvidia NEMO [example](https://developer.nvidia.com/blog/llm-model-pruning-and-knowledge-distillation-with-nvidia-nemo-framework/) with [code](https://github.com/NVIDIA/NeMo/tree/main/tutorials/llm/llama/pruning-distillation)


# Deployment Operations (Containers, Scaling, and Serving)

At this stage, we have focused on how to train and optimize models. Equally important in MLOps is **how to deploy these models as services** in a reliable and scalable manner. This typically involves turning your model into a web service (an API) and using containerization and orchestration technologies to manage that service in production. We will cover Docker and containerization fundamentals, building a model API, using Docker Compose for multi-container setups, and give a high-level overview of Kubernetes for scaling. We’ll also explore a simple FastAPI deployment and discuss tools like BentoML for model serving.

### Docker and Containerization Concepts

**Docker** is a platform that allows you to package an application and its dependencies into a standardized unit called a **container**. Containers are similar to lightweight virtual machines – they provide isolation from the host system, ensuring that your app runs the same anywhere. The key difference between images and containers is: 
- **Docker image** is like a snapshot or blueprint of your application (including the OS environment and software)
- **Docker container** is a running instance of that image. An easy analogy: if an image is a class, then a container is an object (instance of that class); or as CircleCI’s blog puts it, *“An image is a read-only manifest (like a blueprint) of what will be inside the container, and a container is like a shipping container that holds the actual running application”*.

To create a Docker image, we usually write a **Dockerfile** – a text file with instructions on how to set up the environment. For example, a Dockerfile might specify a base image (like `python:3.10`), then copy your code into the image, install required packages, and set a command to run your app. Each instruction in a Dockerfile (like `RUN apt-get install ...` or `COPY . .`) creates a new layer in the image. Docker caches these layers, so rebuilding an image can be very fast if nothing changed in earlier layers (this is why ordering of instructions matters for caching efficiency). The resulting image can be run on any machine that has Docker, and it will contain everything needed to run your model service – from the Python interpreter to the model weights (if you include them).

**Why containers for ML?** In an MLOps setting, Docker is extremely useful for deploying models because it ensures that the environment (library versions, etc.) is consistent between development and production. It also makes scaling and distribution easier – you can ship the container to a cloud server or use Kubernetes to run multiple copies. Containers encapsulate the model, its code (e.g., a Flask or FastAPI server), and dependencies. They also promote immutability: once you have a working image, every container started from it will behave the same, which reduces “it works on my machine” problems.

> **TODO**: docker commands [cheat sheet](https://www.docker.com/resources/cli-cheat-sheet/)

**Dockerfile example:** Let’s say we have a simple FastAPI app (we’ll create one shortly to serve a model). A Dockerfile for it might look like:

```Dockerfile
# Use an official Python base image
FROM python:3.10-slim

# Set working directory
WORKDIR /app

# Install necessary packages
# (It's often good to copy requirements.txt and install before copying the rest of the code for caching)
COPY requirements.txt .
RUN pip install -r requirements.txt

# Copy the app code
COPY . .

# Expose port (if needed, not strictly required)
EXPOSE 80

# Command to run the app
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "80"]
```

If `main:app` is our FastAPI application (meaning in `main.py` we created a FastAPI instance named `app`), this Dockerfile will containerize it. Note how we first copy `requirements.txt` and install dependencies, then copy the rest of the code – this is a common pattern to leverage caching (if code changes but requirements don’t, Docker can reuse the layer with installed packages). We expose port 80 and then run the app with Uvicorn.

We can build this image by running `docker build -t mymodel-api:v1 .` in the directory with the Dockerfile, and then run it with `docker run -p 80:80 mymodel-api:v1`. The app will be accessible on port 80 of the host.

**Model API with FastAPI example:** 

FastAPI is a modern, fast (high-performance), web framework for building APIs with Python 3.7+ based on standard Python type hints. It's known for its high speed and ease of use and has been gaining popularity in the Python community. Here are some key features and benefits of FastAPI:
- Speed: FastAPI is built on Starlette for the web parts and Pydantic for the data parts, which makes it one of the fastest Python frameworks available, only slower than NodeJS and Go according to some benchmarks.
- Automatic Documentation: FastAPI automatically generates interactive API documentation (using Swagger UI and ReDoc) that lets you call and test your API directly from the browser.
- Easy to Use: It has been designed to be easy to use while also ensuring that new developers can quickly understand its operation. FastAPI simplifies the process of building robust APIs.
- Asynchronous Code Support: FastAPI is one of the few Python web frameworks to support asynchronous request handlers out of the box, making it suitable for high I/O-bound applications.
- Security and Authentication: FastAPI includes several tools to help with API security, such as OAuth2 password flow and JWT tokens, as well as utilities for hashing passwords.
- Extensibility: Being lightweight and based on standard Python tools, FastAPI is very easy to extend with various databases, ORMs, authentication and authorization frameworks, data validation libraries, and more.
- Built for Production: It has features and utilities to help deploy and run your applications in production environments, including Docker integration.

To illustrate, here’s a minimal FastAPI app that serves a Hugging Face transformer model (say a text summarization model):

```python
# main.py
from fastapi import FastAPI
from transformers import pipeline

app = FastAPI()
summarizer = pipeline("summarization", model="t5-small")  # load a small T5 model for demo

@app.get("/summarize")
def summarize(text: str):
    summary = summarizer(text, max_length=50, truncation=True)[0]['summary_text']
    return {"summary": summary}
```

This API has one endpoint `/summarize` which takes a query parameter `text` and returns a summarized version. We can run it locally (without Docker) to test:

```bash
uvicorn main:app --reload --port 8000
```

And then hitting `http://localhost:8000/summarize?text=YourTextHere` will return a JSON with the summary.

### Docker Compose and Multi-Container Applications

For deploying ML systems, you often have multiple components: e.g., a web app, a model inference service, a database, maybe a message queue. **Docker Compose** is a tool that helps define and run multi-container applications. With Compose, you write a `docker-compose.yml` file where you list your services (each service corresponds to a container) and their configurations (images, environment variables, network setup, volumes, etc.). Then a single command (`docker-compose up`) can start all of them with the specified configuration.

**Benefits of Docker Compose:** It simplifies coordination of multiple containers. Instead of manually starting each container with the right links and settings, Compose brings up the entire stack in one go. It also makes the environment portable – the YAML file can be shared with others to replicate the exact setup. For instance, if your NLP application consists of a FastAPI model server container and a Postgres database (perhaps storing user queries or model outputs), your Compose file will describe both. Compose will ensure that when you run it, both containers start, the networking is set so that the FastAPI can talk to the database, etc., all with one command. This is great for development and testing, and even for simple deployments on a single host (Compose is essentially limited to managing containers on one host).

An example `docker-compose.yml` (not specific to the above app, but to illustrate structure):

```yaml
version: "3.9"
services:
  api:
    build: .  # build image from Dockerfile in current directory
    ports:
      - "8000:80"
    environment:
      - MODEL_PATH=/models/your_model.bin
  db:
    image: postgres:15
    environment:
      - POSTGRES_USER=user
      - POSTGRES_PASSWORD=secret
    volumes:
      - pgdata:/var/lib/postgresql/data

volumes:
  pgdata:
```

In this file, we have two services: `api` (which we build from local code, presumably the FastAPI app) and `db` (running a PostgreSQL database). We expose the API’s port, set an environment variable (maybe pointing to a model file), and we create a named volume for persistent database storage. With this setup, `docker-compose up` will start both containers; the FastAPI container could use the hostname `db` to connect to the database (Compose sets up a DNS for service names). This greatly simplifies multi-container orchestration – you don’t have to manually run each docker command and link containers.

Compose is especially useful in an MLOps scenario for creating **reproducible environments** for integration tests or staging deployments. For example, you might have a compose file that brings up your model API, a fake data source, and a monitoring agent, to test the whole pipeline locally. It’s also useful for local simulation of what in production might be managed by Kubernetes.

In summary, Docker Compose helps manage multi-container applications with ease, using a single file to configure everything from environment variables to volumes, thereby improving collaboration (everyone can spin up the same environment) and reducing deployment complexity for simple setups.

In [ ]:
!cd fastapi_service
!docker-compose build
!docker-compose up -d

In [21]:
import requests

requests.post('http://0.0.0.0:8088/classify', data="ML deployment is a very complicated and awful process").text

'{"label":"NEGATIVE","score":0.9997016787528992}'

In [22]:
import requests

requests.post('http://0.0.0.0:8088/classify', data="ML deployment is a very simple and exciting process").text

'{"label":"POSITIVE","score":0.9997023940086365}'

# Kubernetes (K8s) for Scaling and Orchestration (high-level overview only)

While Docker and Compose are great for single-machine scenarios, in production we often need to scale across many machines and have higher-level automation for reliability. **Kubernetes** is the de facto platform for container orchestration in the industry. As the official Kubernetes docs say, *“Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications.”*.

What does Kubernetes provide? In essence, Kubernetes allows you to declare the desired state of your deployment (e.g., “run 5 copies of this container, ensure they’re always running, and distribute traffic between them”) and the system works to maintain that state. Key concepts in K8s include:

* **Pods:** A pod is the smallest deployable unit in Kubernetes, typically containing one container (or a few tightly-coupled containers). If you deploy our FastAPI model image to K8s, it will run inside a pod.
* **Deployments:** A Deployment is a Kubernetes object that ensures a certain number of pod replicas are running. You might create a Deployment for your model API with `replicas: 3` to always keep 3 instances. Kubernetes will launch 3 pods and if one crashes, it will automatically replace it with a new one.
* **Services:** In Kubernetes, a Service is an abstraction to expose pods (which have dynamic IPs) under a stable IP or DNS name, and often to load-balance across them. For example, you’d have a Service for the model API, so that requests to the service get distributed to one of the pod replicas.
* **Horizontal Pod Autoscaler (HPA):** This is a component that can automatically increase or decrease the number of pod replicas based on metrics like CPU usage or custom metrics (like request throughput). For an LLM deployment, you might set an HPA to scale out the number of model pods when CPU utilization goes above 70%, and scale back in when it's low, to handle variable traffic.
* **Zero-downtime deployments:** Kubernetes has built-in support for rolling updates. When you update the container image version in a Deployment, it will perform a rolling update – gradually bringing up new pods and bringing down old pods so that the service stays available the whole time. It ensures not all instances are killed at once and can even abort the rollout if health checks fail on the new version, achieving zero (or minimal) downtime upgrades.

![](https://kubernetes.io/images/docs/kubernetes-cluster-architecture.svg)

For example, if we had version 1 of our model and want to deploy version 2, Kubernetes might start one pod with v2 while keeping 3 pods of v1 running, route some traffic to v2, then once v2 is deemed healthy, kill one of the v1 pods and start another v2, and so on (depending on configured strategy). Users consuming the service might not notice any downtime because there’s always at least some pods serving.

Kubernetes also handles things like **self-healing** (if a node in the cluster dies, the pods are rescheduled on other nodes), **scaling nodes** (with Cluster Autoscaler, it can add more VMs to the cluster if needed), and provides an ecosystem for managing configs and secrets, doing A/B deployments (canary, blue-green), and more. It’s a complex but powerful system.

In many MLOps deployments, Kubernetes is used to manage the inference services (and sometimes the training jobs too, via Kubeflow or similar). For instance, you could deploy a whole NLP inference system on K8s: multiple replicas of an LLM API behind a service, an autoscaler watching the request rate, possibly a GPU node pool for the model servers, etc. You would benefit from Kubernetes features like easy scaling, updates, and monitoring.

One should note that running LLMs on Kubernetes requires planning around GPU support (K8s has special resource types for GPUs and you need nodes with GPUs), as well as handling of large model files (using persistent volumes or image bundling). But many organizations successfully serve transformer models in K8s in production.

For our lecture purposes, you don’t need to configure Kubernetes by hand, but it’s important to know what it is. To summarize: Kubernetes is the platform that takes containerized applications to production at scale, handling deployment, scaling, and management tasks automatically. It ensures high availability and scalability, and combined with continuous delivery pipelines, it enables frequent and reliable updates to ML services without downtime.

> **TODO**: k8s architecure [overview](https://kubernetes.io/docs/concepts/architecture/)

<a id='BentoML'></a>
# Simple Approach. BentoML

ML model deployment is the process of integrating a trained ML model into an existing production environment to make practical, actionable decisions based on new data. It's a crucial step in a machine learning project as it allows the model to provide real-world value. Here we will consider **Real-time Inference** - for applications requiring immediate feedback, models are deployed in an environment that supports real-time data processing with a simple implementation using BentoML.

**BentoML** is designed for teams working to bring machine learning (ML) models into production in a reliable, scalable, and cost-efficient way. In particular, AI application developers can leverage BentoML to easily integrate state-of-the-art pre-trained models into their applications. By seamlessly bridging the gap between model creation and production deployment, BentoML promotes collaboration between developers and in-house data science teams.

> **TODO**: read documentations and do some experiments with more advanced options described [here](https://docs.bentoml.org/en/latest/quickstarts/deploy-a-transformer-model-with-bentoml.html)


Developing a FastAPI app from scratch as we did is flexible, but for larger teams and more complex scenarios, specialized model serving frameworks can be helpful. **BentoML** is one such framework that has grown in popularity. BentoML is an open-source platform designed to make it simple to package and deploy machine learning models at scale. It provides an easy way to turn a trained model into a **“Bento” (a model service bundle)** that includes all code, dependencies, and a standard REST API interface.

Key features of BentoML include:

* Support for multiple ML frameworks (PyTorch, TensorFlow, Scikit-learn, XGBoost, etc.), so you can serve models from any of these with a unified approach.
* It generates a production-ready API endpoint for your model with just a few lines of code – you define a `Service` and `Runnable` in BentoML, and it handles creating a Docker image or a deployment from that.
* It has built-in support for **micro-batching**, which means if multiple requests come in at the same time, it can batch them for inference to utilize vectorized operations. This is great for LLMs where a GPU can handle a batch of, say, 4 queries almost as fast as 1 query.
* BentoML offers a CLI and a web UI for managing deployed models, including model versioning and repository. This helps teams collaborate and keeps track of which model versions are deployed where.
* It can deploy to various platforms: Docker containers, Kubernetes, serverless, etc., often with one-line commands or through BentoML’s cloud service. For instance, BentoML can integrate with Kubernetes to handle scaling, or you can use their cloud (BentoCloud) for a hosted solution.

In [ ]:
pip install --upgrade bentoml==1.3.16

In [1]:
import bentoml
import transformers

pipe = transformers.pipeline("text-classification", device='cpu')

bentoml.transformers.save_model(
  "text-classification-pipe",
  pipe,
  signatures={
    "__call__": {"batchable": True}  # Enable dynamic batching for model
  }
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
/tmp/ipykernel_1347848/1918173645.py:6: BentoMLDeprecationWarning: `bentoml.transformers` is deprecated since v1.4 and will be removed in a future version.
  bentoml.transformers.save_model(


Model(tag="text-classification-pipe:mtzbwtbr6ckpoqqb", path="/home/abazdyrev/bentoml/models/text-classification-pipe/mtzbwtbr6ckpoqqb/")

In [2]:
!bentoml models list

 Tag                      Module                Size        Creation Time       
 text-classification-pi…  bentoml.transformers  256.33 MiB  2025-05-16 00:55:00 


In [ ]:
!bentoml serve bentoml_service.py:svc

In [23]:
import requests

requests.post('http://localhost:3000/classify', data="BentoML is awesome").text

'{"label":"POSITIVE","score":0.9998418092727661}'

In [24]:
import requests

requests.post('http://localhost:3000/classify', data="ML deployment is a very complicated and awful process").text

'{"label":"NEGATIVE","score":0.9997016787528992}'

# LLM Deployment Tools (vLLM, llama.cpp)

*llama.cpp* is an open-source project that gained fame for enabling LLM inference (especially Meta’s LLaMA models and others in that family) on commodity hardware like CPUs. Written in C/C++, llama.cpp uses a custom lightweight library (GGML/GGUF) to run models with minimal dependencies. The unique appeal of llama.cpp is that it can run efficiently on just a CPU, or leverage GPUs if available, and even on mobile devices. It uses quantization heavily (more on quantization below) to reduce model memory footprint, allowing, for example, a 7B parameter model to run on a laptop without a dedicated GPU. This democratizes access to LLMs by **eliminating the need for high-end hardware**. Moreover, llama.cpp supports a variety of platforms (Windows, Mac with Apple Silicon, Linux) and can do CPU–GPU hybrid inference (load what fits on GPU, rest on CPU). Many community models are distributed in llama.cpp’s format, which uses the GGUF container for quantized weights. For developers, llama.cpp provides a simple CLI and also has bindings for Python (through `llama-cpp-python`) to integrate into applications. It’s ideal for deploying LLMs in resource-constrained environments or when simplicity and portability are priorities.

from https://huggingface.co/bartowski/gemma-2-2b-it-GGUF

In [3]:
!huggingface-cli download bartowski/gemma-2-2b-it-GGUF --include "gemma-2-2b-it-Q4_K_M.gguf" --local-dir /home/abazdyrev/pretrained_models

Fetching 1 files:   0%|                                   | 0/1 [00:00<?, ?it/s]Downloading 'gemma-2-2b-it-Q4_K_M.gguf' to '/home/abazdyrev/pretrained_models/.cache/huggingface/download/Lm_sEH_Lw9SnTwhv8NL-PxqbHbw=.e0aee85060f168f0f2d8473d7ea41ce2f3230c1bc1374847505ea599288a7787.incomplete'
Xet Storage is enabled for this repo. Downloading file from Xet Storage..

gemma-2-2b-it-Q4_K_M.gguf:   0%|                    | 0.00/1.71G [00:00<?, ?B/s]
gemma-2-2b-it-Q4_K_M.gguf:   0%|           | 131k/1.71G [00:00<3:26:04, 138kB/s]
gemma-2-2b-it-Q4_K_M.gguf:   0%|           | 5.72M/1.71G [00:01<03:50, 7.39MB/s]
gemma-2-2b-it-Q4_K_M.gguf:   1%|▏          | 25.4M/1.71G [00:01<00:51, 32.5MB/s]
gemma-2-2b-it-Q4_K_M.gguf:   3%|▎          | 48.4M/1.71G [00:01<00:42, 38.9MB/s]
gemma-2-2b-it-Q4_K_M.gguf:   5%|▌          | 85.2M/1.71G [00:01<00:20, 77.5MB/s]
gemma-2-2b-it-Q4_K_M.gguf:   9%|█▏           | 154M/1.71G [00:02<00:10, 148MB/s]
gemma-2-2b-it-Q4_K_M.gguf:  10%|█▎           | 177M/1.71G [00:02<0

```bash
docker pull ghcr.io/abetlen/llama-cpp-python:latest
docker run --rm -it -p 8000:8000 -d -v /home/abazdyrev/pretrained_models:/models \
-e MODEL=/models/gemma-2-2b-it-Q4_K_M.gguf \
ghcr.io/abetlen/llama-cpp-python:latest
```

* `docker run`
  Starts a new container from the given image.

* `--rm`
  Automatically remove (“clean up”) the container filesystem when the container exits. Prevents stopped containers from piling up.

* `-it`
  Combines:

  * `-i` (interactive): Keep STDIN open even if not attached.
  * `-t` (tty): Allocate a pseudo-TTY, so you can interact via a shell if the container exposes one.

* `-p 8000:8000`
  Publish container port **8000** to the host’s port **8000**, so you can reach whatever service the container is serving on port 8000 (e.g. an HTTP API) at `localhost:8000`.

* `-d`
  Run the container “detached” in the background. Docker will print the new container ID and return you to your shell prompt immediately.

* `-v /home/abazdyrev/pretrained_models:/models`
  Mount a host directory into the container’s filesystem:

  * Host path: `/home/abazdyrev/pretrained_models`
  * Container path: `/models`
    This makes your pretrained GGUF files (and any others in that folder) accessible inside the container under `/models`.

* `-e MODEL=/models/gemma-2-2b-it-Q4_K_M.gguf`
  Set an environment variable inside the container:

  * `MODEL` → `/models/gemma-2-2b-it-Q4_K_M.gguf`
    The llama-cpp-python entrypoint will read this and load that specific model file at startup.

In [25]:
from openai import OpenAI
openai_api_base = "http://localhost:8000/v1"
openai_api_key = "not-needed"



messages = [{
    "role": "user",
    "content": "Hello! I need you to write an essay (up to 200 words) about the future of the AI for the next 10 years."
}]


client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

response = client.models.list()
print(response)


model="/models/gemma-2-2b-it-Q4_K_M.gguf"
max_tokens=512
response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_tokens=max_tokens,
    temperature=0.2,  # low temperature for more deterministic output
    stop=None
)
response

SyncPage[Model](data=[Model(id='/models/gemma-2-2b-it-Q4_K_M.gguf', created=None, object='model', owned_by='me', permissions=[])], object='list')


ChatCompletion(id='chatcmpl-184f69aa-44d5-4196-b501-0393046797b0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="The next decade promises a transformative era for AI, driven by rapid advancements in computing power and data availability. We'll see AI systems become increasingly sophisticated, capable of complex tasks like creative writing, scientific discovery, and personalized healthcare. \n\nExpect to see AI integrated into everyday life, from smart homes and autonomous vehicles to personalized education and financial management.  However, ethical considerations will remain paramount. Bias in algorithms, data privacy concerns, and the potential for job displacement will require careful navigation. \n\nThe focus will shift towards responsible development and deployment of AI, ensuring its benefits are shared equitably and its risks are mitigated. This decade will be a period of rapid evolution, laying the foundation for a future w

**vLLM:** *vLLM* is a high-performance inference engine tailored for serving large language models. Originally developed at UC Berkeley, it has become a go-to solution for production LLM serving due to its throughput and memory optimizations. vLLM introduces an optimized attention mechanism called **PagedAttention**, which manages the model’s key/value caches more efficiently in memory. It also supports **continuous batching** of incoming requests, meaning it can dynamically batch together multiple user queries and process them in one forward pass to maximize GPU utilization. In practice, vLLM can significantly increase total throughput (requests per second) compared to naive serving. Another advantage is that vLLM provides an **OpenAI-compatible API server**, so you can spin up your own service with the same API as OpenAI’s GPT-4, but backed by a local model – useful for integration into existing applications. It supports a range of hardware (NVIDIA and AMD GPUs, as well as CPU execution) and even integrates optimizations like **FlashAttention** for faster transformer computations. In summary, vLLM is designed to make LLM inference *easy, fast, and cheap* by combining state-of-the-art optimizations in a single package.

In [ ]:
docker pull vllm/vllm-openai:latest
docker run --gpus all -p 8000:8000 --ipc=host \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    vllm/vllm-openai:latest \
    --model TechxGenus/gemma-2b-it-AWQ \
    --gpu-memory-utilization 0.1

In [27]:
from openai import OpenAI
openai_api_base = "http://localhost:8000/v1"
openai_api_key = "not-needed"



messages = [{
    "role": "user",
    "content": "Hello! I need you to write an essay (up to 200 words) about the future of the AI for the next 10 years."
}]


client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

response = client.models.list()
print(response)


model="TechxGenus/gemma-2b-it-AWQ"
max_tokens=512
response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_tokens=max_tokens,
    temperature=0.2,  # low temperature for more deterministic output
    stop=None
)
response

SyncPage[Model](data=[Model(id='TechxGenus/gemma-2b-it-AWQ', created=1747398181, object='model', owned_by='vllm', root='TechxGenus/gemma-2b-it-AWQ', parent=None, max_model_len=8192, permission=[{'id': 'modelperm-74cdc67e6bfa4fe7a58ebcfb0e5102a4', 'object': 'model_permission', 'created': 1747398181, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}])], object='list')


ChatCompletion(id='chatcmpl-50b67587304644b7927f25a256e7b76d', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="The future of AI for the next 10 years is a complex and fascinating topic. AI will continue to evolve and become more sophisticated, but it will also face challenges such as bias, privacy, and ethical concerns.\n\nOne of the most important challenges is bias. AI systems will be trained on data, and if that data contains bias, the AI system will learn that bias. This can lead to AI systems that are unfair or discriminatory.\n\nAnother challenge is privacy. AI systems will collect and store large amounts of data, and if that data is not properly secured, it could be compromised. This could lead to AI systems that are unable to operate properly.\n\nEthical concerns are also a major challenge. AI systems will be able to make decisions that affect people's lives, and if those decisions are not made properly, it could lead to AI s

## Summary  
`vLLM` and `llama.cpp` serve similar purposes—running local LLMs behind an OpenAI-compatible API—but they are optimised for very different constraints. **vLLM** maximises multi-user throughput on NVIDIA GPUs via its PagedAttention kernel and continuous batching, while **llama.cpp** focuses on extreme portability and memory frugality, allowing quantised GGUF models to run on anything from CPUs to Apple Silicon and even phones. Choose **vLLM** when you have a beefy GPU farm and need speed; choose **llama.cpp** when you need to fit large models onto modest or heterogeneous hardware.


## 1&nbsp;· Architectural foundations  

### 1.1  vLLM  
* **Backend** – C++/CUDA kernels wrapped by PyTorch; employs a *paged* KV-cache that mimics virtual memory, virtually eliminating fragmentation and enabling cache sharing between requests. 
* **Batching** – *continuous batching* lets new requests join an active batch without flushing it, yielding up to $23\times$ higher tokens/s than static batching.

### 1.2  llama.cpp  
* **Backend** – pure C/C++ on top of **ggml**; runs on CPU by default but can off-load layers to CUDA, Metal, or Vulkan.  
* **Quantisation pipeline** – ships several post-training quantisers (e.g. Q4_K_M, IQ2_X) that shrink weights 8–16× while retaining ≈95 % perplexity.  
* **GGUF format** – self-describing single-file container that stores quantised tensors, tokenizer, and metadata for easy distribution.



## 2&nbsp;· Deployment & server interfaces  

| Feature | **vLLM** | **llama.cpp** |
|---------|----------|---------------|
| Launch one-liner | `vllm serve MODEL` or Docker image `vllm/vllm-openai`  | `python -m llama_cpp.server` or compiled binary; Docker images on GHCR  |
| API compatibility | Chat / Completions / Embeddings | Chat / Completions |
| Streaming | ✔ (Server-Sent Events) | ✔ (SSE) |
| Multi-GPU | data- or tensor-parallel | not yet (single GPU or CPU) |


## 3&nbsp;· Hardware compatibility  

|  | vLLM | llama.cpp |
|--|------|-----------|
| **CPU-only** | ✖ (fallback only) | ✔ (AVX2, ARM NEON) |
| **NVIDIA CUDA** | ✔ (≥ SM 7.0) | ✔ (optional) |
| **Apple Metal** | ✖ | ✔ (M-series) [^4] |
| **Mobile** | ✖ | ✔ (Vulkan / CoreML) |
| **RAM/VRAM envelope** | ≥ 2× FP16 model size + cache | as low as model-on-disk + few hundred MB |
